In [1]:
# 01. IMPORT LIBRARIES

import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# 02. PROJECT PATHS

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEANED_TRACKS = PROJECT_ROOT / "data" / "interim" / "cleaned_tracks"
CLEANED_SATELLITE = PROJECT_ROOT / "data" / "interim" / "cleaned_satellite"
FUSED_DIR = PROJECT_ROOT / "data" / "processed" / "fused_dataset"

FUSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Fused data folder:", FUSED_DIR)

Project root: e:\SIH\tropical-cyclone-ai
Fused data folder: e:\SIH\tropical-cyclone-ai\data\processed\fused_dataset


In [3]:
# 03. LOAD FUSED DATASET

FUSED_FILE = FUSED_DIR / "tcir_ibtracs_fused.csv"

fused_df = pd.read_csv(FUSED_FILE)

print("Shape:", fused_df.shape)
display(fused_df.head())

Shape: (21076, 15)


,REGION,CYCLONE_ID,LON_TCIR,LAT_TCIR,TIME,WIND_KTS,SIZE,PRESSURE_MB,USA_ATCF_ID,ISO_TIME,SID,LAT_IBTRACS,LON_IBTRACS,USA_WIND,USA_PRES
0,ATLN,200301L,-66.0,31.4,2003-04-18 15:00:00,30.0,0.0,1008.0,AL012003,2003-04-18 15:00:00,2003108N29294,31.3,-66.0,30,1008
1,ATLN,200301L,-66.3,31.9,2003-04-18 18:00:00,30.0,0.0,1007.0,AL012003,2003-04-18 18:00:00,2003108N29294,31.9,-66.3,30,1007
2,ATLN,200301L,-66.6,32.5,2003-04-18 21:00:00,30.0,0.0,1007.0,AL012003,2003-04-18 21:00:00,2003108N29294,32.5,-66.6,30,1007
3,ATLN,200301L,-68.6,34.5,2003-04-19 12:00:00,35.0,0.0,1006.0,AL012003,2003-04-19 12:00:00,2003108N29294,34.5,-68.6,35,1006
4,ATLN,200301L,-68.8,34.4,2003-04-19 15:00:00,35.0,0.0,1006.0,AL012003,2003-04-19 15:00:00,2003108N29294,34.5,-68.9,35,1006


In [4]:
# 04. CHECK COLUMNS

print("Columns:")
print(fused_df.columns.tolist())

Columns:
['REGION', 'CYCLONE_ID', 'LON_TCIR', 'LAT_TCIR', 'TIME', 'WIND_KTS', 'SIZE', 'PRESSURE_MB', 'USA_ATCF_ID', 'ISO_TIME', 'SID', 'LAT_IBTRACS', 'LON_IBTRACS', 'USA_WIND', 'USA_PRES']


In [5]:
# 05. CHECK MISSING VALUES

missing = fused_df.isna().sum()

print("Total missing values:", missing.sum())
display(missing[missing > 0])

Total missing values: 0


Series([], dtype: int64)

In [6]:
# 06. CHECK CYCLONE COVERAGE

print("Unique cyclones:", fused_df["CYCLONE_ID"].nunique())
print("\nRegions:")
print(fused_df["REGION"].value_counts())

Unique cyclones: 485

Regions:
REGION
WPAC    8922
ATLN    7144
EPAC    5010
Name: count, dtype: int64


In [7]:
# 07. CHECK IMAGE-METADATA ALIGNMENT

print("Fused records:", len(fused_df))
print("Expected TCIR images:", 21076)
print("Alignment:", len(fused_df) == 21076)

Fused records: 21076
Expected TCIR images: 21076
Alignment: True


In [8]:
# 08. CHECK IBTRACS MATCHING

print("IBTrACS matches:", fused_df["SID"].notna().sum())
print("Unmatched records:", fused_df["SID"].isna().sum())

IBTrACS matches: 21076
Unmatched records: 0


In [9]:
# 09. CHECK TIME RANGE

print("Start:", fused_df["TIME"].min())
print("End:", fused_df["TIME"].max())

Start: 2003-01-11 18:00:00
End: 2016-12-28 06:00:00


In [10]:
# 10. CHECK DUPLICATE RECORDS

duplicates = fused_df.duplicated(
    subset=["CYCLONE_ID", "TIME"],
    keep=False
)

print("Duplicate cyclone-time records:", duplicates.sum())

Duplicate cyclone-time records: 21076


In [11]:
# 11. COMPARE LOCATIONS

fused_df["LAT_DIFF"] = fused_df["LAT_TCIR"] - fused_df["LAT_IBTRACS"]

fused_df["LON_TCIR_NORM"] = ((fused_df["LON_TCIR"] + 180) % 360) - 180
fused_df["LON_IBTRACS_NORM"] = ((fused_df["LON_IBTRACS"] + 180) % 360) - 180

fused_df["LON_DIFF"] = (
    fused_df["LON_TCIR_NORM"] - fused_df["LON_IBTRACS_NORM"]
)

print("Latitude difference:")
display(fused_df["LAT_DIFF"].describe())

print("\nLongitude difference:")
display(fused_df["LON_DIFF"].describe())

Latitude difference:


count    21076.000000
mean         0.011596
std          0.115865
min         -2.300000
25%          0.000000
50%          0.000000
75%          0.000000
max          2.000000
Name: LAT_DIFF, dtype: float64


Longitude difference:


count    21076.000000
mean        -0.064623
std          4.923523
min       -359.700000
25%          0.000000
50%          0.000000
75%          0.000000
max          3.200000
Name: LON_DIFF, dtype: float64

In [12]:
# 12. COMPARE INTENSITY

fused_df["WIND_DIFF"] = fused_df["WIND_KTS"] - fused_df["USA_WIND"]
fused_df["PRESSURE_DIFF"] = fused_df["PRESSURE_MB"] - fused_df["USA_PRES"]

print("Wind difference:")
display(fused_df["WIND_DIFF"].describe())

print("\nPressure difference:")
display(fused_df["PRESSURE_DIFF"].describe())

Wind difference:


count    21076.000000
mean        -0.038148
std          0.403151
min        -21.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          5.000000
Name: WIND_DIFF, dtype: float64


Pressure difference:


count    21076.000000
mean        -0.020307
std          0.298300
min         -3.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         15.000000
Name: PRESSURE_DIFF, dtype: float64

In [13]:
# 13. FINAL FUSION PREVIEW

display(fused_df[[
    "REGION",
    "CYCLONE_ID",
    "USA_ATCF_ID",
    "TIME",
    "LAT_TCIR",
    "LON_TCIR",
    "LAT_IBTRACS",
    "LON_IBTRACS",
    "WIND_KTS",
    "PRESSURE_MB",
    "USA_WIND",
    "USA_PRES"
]].head(10))

,REGION,CYCLONE_ID,USA_ATCF_ID,TIME,LAT_TCIR,LON_TCIR,LAT_IBTRACS,LON_IBTRACS,WIND_KTS,PRESSURE_MB,USA_WIND,USA_PRES
0,ATLN,200301L,AL012003,2003-04-18 15:00:00,31.4,-66.0,31.3,-66.0,30.0,1008.0,30,1008
1,ATLN,200301L,AL012003,2003-04-18 18:00:00,31.9,-66.3,31.9,-66.3,30.0,1007.0,30,1007
2,ATLN,200301L,AL012003,2003-04-18 21:00:00,32.5,-66.6,32.5,-66.6,30.0,1007.0,30,1007
3,ATLN,200301L,AL012003,2003-04-19 12:00:00,34.5,-68.6,34.5,-68.6,35.0,1006.0,35,1006
4,ATLN,200301L,AL012003,2003-04-19 15:00:00,34.4,-68.8,34.5,-68.9,35.0,1006.0,35,1006
5,ATLN,200301L,AL012003,2003-04-19 18:00:00,34.3,-69.1,34.3,-69.1,35.0,1006.0,35,1006
6,ATLN,200301L,AL012003,2003-04-19 21:00:00,34.0,-69.0,34.0,-69.1,38.0,1006.0,38,1006
7,ATLN,200301L,AL012003,2003-04-20 12:00:00,32.0,-68.2,32.0,-68.2,45.0,1000.0,45,1000
8,ATLN,200301L,AL012003,2003-04-20 15:00:00,31.9,-67.8,31.8,-67.8,45.0,999.0,45,999
9,ATLN,200301L,AL012003,2003-04-20 18:00:00,31.7,-67.3,31.7,-67.3,45.0,998.0,45,998


In [14]:
# 14. FUSION SUMMARY

print("TCIR records:", len(fused_df))
print("IBTrACS matches:", fused_df["SID"].notna().sum())
print("Unmatched records:", fused_df["SID"].isna().sum())
print("Unique cyclones:", fused_df["CYCLONE_ID"].nunique())
print("Final fusion status:", "SUCCESS" if (
    len(fused_df) == 21076 and
    fused_df["SID"].isna().sum() == 0
) else "CHECK REQUIRED")

TCIR records: 21076
IBTrACS matches: 21076
Unmatched records: 0
Unique cyclones: 485
Final fusion status: SUCCESS


In [15]:
# 15. PREPARE FINAL FUSED DATASET

final_fused_df = fused_df.drop(
    columns=[
        "LAT_DIFF",
        "LON_TCIR_NORM",
        "LON_IBTRACS_NORM",
        "LON_DIFF",
        "WIND_DIFF",
        "PRESSURE_DIFF"
    ],
    errors="ignore"
).copy()

print("Final shape:", final_fused_df.shape)

Final shape: (21076, 15)


In [17]:
# 16. SAVE FINAL FUSED DATASET

final_fused_df.to_csv(
    FUSED_FILE,
    index=False
)

print("Saved:", FUSED_FILE)
print("Final shape:", final_fused_df.shape)

Saved: e:\SIH\tropical-cyclone-ai\data\processed\fused_dataset\tcir_ibtracs_fused.csv
Final shape: (21076, 15)


In [18]:
# 17. FINAL VERIFICATION

print("Final rows:", len(final_fused_df))
print("Final columns:", len(final_fused_df.columns))
print("IBTrACS unmatched:", final_fused_df["SID"].isna().sum())
print("Unique cyclones:", final_fused_df["CYCLONE_ID"].nunique())
print("Missing values:", final_fused_df.isna().sum().sum())

Final rows: 21076
Final columns: 15
IBTrACS unmatched: 0
Unique cyclones: 485
Missing values: 0


## 18. DATA FUSION SUMMARY

### Fusion Results

- TCIR satellite images: **21,076**
- TCIR metadata records: **21,076**
- Global IBTrACS records matched: **21,076**
- Unmatched records: **0**
- Unique cyclones: **485**
- Final fused dataset: **21,076 rows × 15 columns**

### Fusion Method

TCIR satellite-image metadata was matched with global IBTrACS cyclone-track data using the **cyclone identifier and observation timestamp**.

The final fused dataset combines satellite-image information with corresponding cyclone track and intensity information and is ready for downstream cyclone identification, classification, and prediction tasks.